In [ ]:
!pip install -q sentence-transformers faiss-cpu pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 112.9 MB/s eta 0:00:00


In [ ]:
import os
import re
import ast
import gc
import json
import math
import time
import warnings
import unicodedata
from pathlib import Path
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd

import faiss
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 220)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
LOCAL_FAISS_DIR = Path("artifacts/faiss")
DRIVE_FAISS_DIR = Path("/content/drive/MyDrive/tablewise/artifacts_new/faiss")

if DRIVE_FAISS_DIR.exists():
    FAISS_DIR = DRIVE_FAISS_DIR
elif LOCAL_FAISS_DIR.exists():
    FAISS_DIR = LOCAL_FAISS_DIR
else:
    raise FileNotFoundError(
        "Nu am găsit artifacts/faiss."
    )

MAPPING_PATH = FAISS_DIR / "restaurant_index_mapping.parquet"
EMBEDDINGS_PATH = FAISS_DIR / "restaurant_embeddings.npy"
FAISS_INDEX_PATH = FAISS_DIR / "restaurant_faiss.index"
METADATA_PATH = FAISS_DIR / "metadata.json"

OUTPUT_DIR = FAISS_DIR.parent / "reranking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("FAISS_DIR:", FAISS_DIR)
print("MAPPING_PATH:", MAPPING_PATH)
print("EMBEDDINGS_PATH:", EMBEDDINGS_PATH)
print("FAISS_INDEX_PATH:", FAISS_INDEX_PATH)
print("METADATA_PATH:", METADATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

FAISS_DIR: /content/drive/MyDrive/tablewise/artifacts_new/faiss
MAPPING_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet
EMBEDDINGS_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy
FAISS_INDEX_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_faiss.index
METADATA_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/metadata.json
OUTPUT_DIR: /content/drive/MyDrive/tablewise/artifacts_new/reranking


In [ ]:
assert MAPPING_PATH.exists(), f"Lipsește mapping-ul: {MAPPING_PATH}"
assert EMBEDDINGS_PATH.exists(), f"Lipsesc embeddings: {EMBEDDINGS_PATH}"
assert FAISS_INDEX_PATH.exists(), f"Lipsește FAISS index: {FAISS_INDEX_PATH}"

mapping_df = pd.read_parquet(MAPPING_PATH)
embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

if METADATA_PATH.exists():
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        faiss_metadata = json.load(f)
else:
    faiss_metadata = {}

print("mapping_df shape:", mapping_df.shape)
print("embeddings shape:", embeddings.shape)
print("FAISS ntotal:", faiss_index.ntotal)

display(mapping_df.head(3))
faiss_metadata

mapping_df shape: (1033798, 31)
embeddings shape: (1033798, 384)
FAISS ntotal: 1033798


,faiss_idx,restaurant_id,name,country,region,province,city,city_filled,city_filled_norm,city_source,address,latitude,longitude,rating,price_level,price_bucket,top_tags_text,top_tags_norm_list,meals_text,meals_norm_list,features_text,features_norm_list,special_diets_text,special_diets_norm_list,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score,profile_quality_score,short_profile,profile_text
0,0,g10001637-d10002227,Le 147,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,4.0,€,cheap,"Cheap Eats, French","[cheap eats, french]","Lunch, Dinner","[lunch, dinner]","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service","[reservations, seating, wheelchair accessible, serves alcohol, accepts credit cards, table service]",Unknown,[],#1 of 2 Restaurants in Saint-Jouvent,1.0,2.0,0.36907,11,"Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French","Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Featu..."
1,1,g10001637-d14975787,Le Saint Jouvent,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,4.0,€,cheap,Cheap Eats,[cheap eats],Unknown,[],Unknown,[],Unknown,[],#2 of 2 Restaurants in Saint-Jouvent,2.0,2.0,0.00000,9,"Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats","Le Saint Jouvent is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 16 Place de l Eglise, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats. Popularity: #2 of 2 Restaur..."
2,2,g10002858-d4586832,Au Bout du Pont,France,Centre-Val de Loire,Berry,Rivarennes,Rivarennes,rivarennes,original_city,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,5.0,€,cheap,"Cheap Eats, French, European","[cheap eats, french, european]","Dinner, Lunch, Drinks","[dinner, lunch, drinks]","Reservations, Seating, Table Service, Wheelchair Accessible","[reservations, seating, table service, wheelchair accessible]",Unknown,[],#1 of 1 Restaurant in Rivarennes,1.0,1.0,NaN,11,"Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European","Au Bout du Pont is a restaurant located in Rivarennes, Berry, Centre-Val de Loire, France. Address: 2 rue des Dames, 36800 Rivarennes France. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French, European. Meals served: Dinner, Lunch..."


{'created_at': '2026-05-04T16:00:35.667690Z',
 'input_path': '/content/drive/MyDrive/tablewise/data_new/processed/restaurants_indexable.parquet',
 'embedding_data_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurants_for_embeddings.parquet',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'num_restaurants': 1033798,
 'embedding_dim': 384,
 'normalized_embeddings': True,
 'faiss_index_type': 'IndexFlatIP',
 'use_sample': False,
 'sample_size': None,
 'embeddings_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy',
 'faiss_index_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_faiss.index',
 'mapping_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet',
 'python_version': '3.12.13',
 'numpy_version': '2.0.2',
 'pandas_version': '2.2.2'}

In [ ]:
required_cols = [
    "faiss_idx",
    "restaurant_id",
    "name",
    "country",
    "city_filled",
    "city_filled_norm",
    "price_bucket",
    "rating",
    "popularity_score",
    "profile_text",
    "short_profile",
]

missing_cols = [c for c in required_cols if c not in mapping_df.columns]
assert not missing_cols, f"Lipsesc coloane obligatorii din mapping: {missing_cols}"

assert len(mapping_df) == embeddings.shape[0], "mapping_df și embeddings au dimensiuni diferite."
assert len(mapping_df) == faiss_index.ntotal, "mapping_df și FAISS index au dimensiuni diferite."
assert mapping_df["faiss_idx"].is_unique, "faiss_idx nu este unic."
assert mapping_df["faiss_idx"].min() == 0, "faiss_idx nu începe de la 0."
assert mapping_df["faiss_idx"].max() == len(mapping_df) - 1, "faiss_idx nu este continuu."

print("Validări OK.")

Validări OK.


In [ ]:
EMBEDDING_MODEL_NAME = faiss_metadata.get(
    "embedding_model",
    "sentence-transformers/all-MiniLM-L6-v2",
)

print("Loading embedding model:", EMBEDDING_MODEL_NAME)

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Model loaded.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.


In [ ]:
def strip_accents(text):
    if text is None or pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    return text


def normalize_text(x):
    if x is None or pd.isna(x):
        return ""

    s = strip_accents(str(x)).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


def safe_value(x, default="Unknown"):
    if x is None or pd.isna(x):
        return default

    s = str(x).strip()

    if not s or s.lower() in {"nan", "none", "null"}:
        return default

    return s


def contains_phrase(text_norm, phrase_norm):
    if not text_norm or not phrase_norm:
        return False

    pattern = rf"(?<![a-z0-9]){re.escape(phrase_norm)}(?![a-z0-9])"
    return re.search(pattern, text_norm) is not None

In [ ]:
def parse_possible_list(x):
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, np.ndarray):
        values = x.tolist()
    elif isinstance(x, (list, tuple, set)):
        values = list(x)
    else:
        s = str(x).strip()

        if not s or s.lower() in {"nan", "none", "null", "unknown"}:
            return []

        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    values = list(parsed)
                else:
                    values = [s]
            except Exception:
                values = re.split(r"[,|;/]", s)
        else:
            values = re.split(r"[,|;/]", s)

    cleaned = []
    seen = set()

    for item in values:
        item_s = safe_value(item, default="").strip()
        item_norm = normalize_text(item_s)

        if item_norm and item_norm not in seen:
            cleaned.append(item_s)
            seen.add(item_norm)

    return cleaned

In [ ]:
df = mapping_df.copy()

list_base_cols = [
    "top_tags",
    "meals",
    "features",
    "special_diets",
]

for base in list_base_cols:
    text_col = f"{base}_text"
    list_col = f"{base}_list"
    norm_list_col = f"{base}_norm_list"

    if list_col not in df.columns:
        if text_col in df.columns:
            df[list_col] = df[text_col].apply(parse_possible_list)
        else:
            df[list_col] = [[] for _ in range(len(df))]
    else:
        df[list_col] = df[list_col].apply(parse_possible_list)

    if norm_list_col not in df.columns:
        df[norm_list_col] = df[list_col].apply(
            lambda xs: [normalize_text(x) for x in xs if normalize_text(x)]
        )
    else:
        df[norm_list_col] = df[norm_list_col].apply(parse_possible_list)
        df[norm_list_col] = df[norm_list_col].apply(
            lambda xs: [normalize_text(x) for x in xs if normalize_text(x)]
        )

for col in ["country", "region", "province", "city", "city_filled"]:
    if col in df.columns:
        df[f"{col}_norm"] = df[col].apply(normalize_text)

df["city_filled_norm"] = df["city_filled"].apply(normalize_text)
df["country_norm"] = df["country"].apply(normalize_text)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["popularity_score"] = pd.to_numeric(df["popularity_score"], errors="coerce")

print("Prepared df shape:", df.shape)
display(df[["name", "country", "city_filled", "rating", "price_bucket", "popularity_score", "top_tags_norm_list"]].head(3))

Prepared df shape: (1033798, 39)


,name,country,city_filled,rating,price_bucket,popularity_score,top_tags_norm_list
0,Le 147,France,Saint-Jouvent,4.0,cheap,0.36907,"[cheap eats, french]"
1,Le Saint Jouvent,France,Saint-Jouvent,4.0,cheap,0.00000,[cheap eats]
2,Au Bout du Pont,France,Rivarennes,5.0,cheap,NaN,"[cheap eats, french, european]"


In [ ]:
BAD_CITY_TERMS = {
    "",
    "nan",
    "none",
    "null",
    "unknown",
    "street",
    "st",
    "road",
    "rd",
    "avenue",
    "ave",
    "square",
    "center",
    "centre",
    "park",
    "hotel",
    "restaurant",
    "restaurants",
    "best",
    "cheap",
    "budget",
    "expensive",
    "food",
    "pub",
    "bar",
    "cafe",
    "cafes",
    "coffee",
    "pizza",
    "pasta",
    "sushi",
    "seafood",
    "breakfast",
    "brunch",
    "lunch",
    "dinner",
    "europe",
    "united kingdom uk",
    "united kingdom",
    "uk",
    "england",
    "scotland",
    "wales",
    "northern ireland",
    "france",
    "italy",
    "spain",
    "germany",
    "portugal",
    "greece",
}

country_vocab = sorted(
    {
        normalize_text(x)
        for x in df["country_norm"].dropna().tolist()
        if normalize_text(x) not in {"", "unknown", "nan", "none", "null"}
    },
    key=len,
    reverse=True,
)

country_values = set(df["country_norm"].dropna().astype(str).map(normalize_text))
region_values = set(df["region_norm"].dropna().astype(str).map(normalize_text)) if "region_norm" in df.columns else set()
province_values = set(df["province_norm"].dropna().astype(str).map(normalize_text)) if "province_norm" in df.columns else set()

blocked_geo_values = country_values | region_values | province_values | BAD_CITY_TERMS
city_counts = df["city_filled_norm"].value_counts(dropna=True).to_dict()

raw_city_values = (
    df["city_filled_norm"]
    .dropna()
    .astype(str)
    .map(normalize_text)
    .tolist()
)

city_vocab = []

for city in sorted(set(raw_city_values), key=len, reverse=True):
    if not city:
        continue
    if city in blocked_geo_values:
        continue
    if len(city) < 2:
        continue
    if city.isdigit():
        continue
    if len(city.split()) > 6:
        continue

    city_vocab.append(city)

IMPORTANT_CITY_ALIASES_RAW = {
    "rome": ["rome", "roma"],
    "paris": ["paris"],
    "madrid": ["madrid"],
    "barcelona": ["barcelona"],
    "milan": ["milan", "milano"],
    "berlin": ["berlin"],
    "lisbon": ["lisbon", "lisboa"],
    "athens": ["athens", "athina", "athena"],
    "london": ["london", "londra"],
    "manchester": ["manchester"],
    "edinburgh": ["edinburgh"],
    "dublin": ["dublin"],
    "vienna": ["vienna", "wien"],
    "prague": ["prague", "praha"],
    "amsterdam": ["amsterdam"],
    "brussels": ["brussels", "bruxelles", "brussel"],
    "budapest": ["budapest"],
    "warsaw": ["warsaw", "warszawa"],
    "krakow": ["krakow", "kraków"],
    "porto": ["porto", "oporto"],
    "florence": ["florence", "firenze"],
    "venice": ["venice", "venezia"],
    "naples": ["naples", "napoli"],
    "turin": ["turin", "torino"],
    "seville": ["seville", "sevilla"],
    "valencia": ["valencia"],
    "granada": ["granada"],
    "malaga": ["malaga", "málaga"],
    "nice": ["nice"],
    "lyon": ["lyon"],
    "marseille": ["marseille"],
    "bordeaux": ["bordeaux"],
    "hamburg": ["hamburg"],
    "munich": ["munich", "muenchen", "münchen"],
    "cologne": ["cologne", "koln", "köln"],
    "frankfurt": ["frankfurt"],
}

CITY_ALIASES = {}

for canonical, aliases in IMPORTANT_CITY_ALIASES_RAW.items():
    canonical_norm = normalize_text(canonical)
    alias_norms = [normalize_text(a) for a in aliases]

    if canonical_norm not in city_vocab:
        city_vocab.append(canonical_norm)

    for alias_norm in alias_norms:
        CITY_ALIASES[alias_norm] = canonical_norm

important_order = list(IMPORTANT_CITY_ALIASES_RAW.keys())

city_vocab = sorted(
    set(city_vocab),
    key=lambda c: (
        0 if c in important_order else 1,
        important_order.index(c) if c in important_order else 999999,
        -len(c),
    ),
)

print("Countries:", len(country_vocab))
print("Cities:", len(city_vocab))

print("\nCity checks:")
for c in ["rome", "paris", "madrid", "barcelona", "milan", "berlin", "lisbon", "athens", "london"]:
    print(c, "in vocab:", c in city_vocab, "| rows:", int((df["city_filled_norm"] == c).sum()))

Countries: 24
Cities: 62251

City checks:
rome in vocab: True | rows: 12603
paris in vocab: True | rows: 18126
madrid in vocab: True | rows: 12133
barcelona in vocab: True | rows: 10283
milan in vocab: True | rows: 8381
berlin in vocab: True | rows: 0
lisbon in vocab: True | rows: 5261
athens in vocab: True | rows: 2915
london in vocab: True | rows: 0


In [ ]:
def flatten_norm_list_column(series):
    values = Counter()

    for xs in series:
        if not isinstance(xs, list):
            continue

        for x in xs:
            x_norm = normalize_text(x)

            if x_norm:
                values[x_norm] += 1

    return values


tag_counter = flatten_norm_list_column(df["top_tags_norm_list"]) if "top_tags_norm_list" in df.columns else Counter()
meal_counter = flatten_norm_list_column(df["meals_norm_list"]) if "meals_norm_list" in df.columns else Counter()
feature_counter = flatten_norm_list_column(df["features_norm_list"]) if "features_norm_list" in df.columns else Counter()
diet_counter = flatten_norm_list_column(df["special_diets_norm_list"]) if "special_diets_norm_list" in df.columns else Counter()

tag_vocab = sorted(tag_counter.keys(), key=len, reverse=True)
meal_vocab = sorted(meal_counter.keys(), key=len, reverse=True)
feature_vocab = sorted(feature_counter.keys(), key=len, reverse=True)
diet_vocab = sorted(diet_counter.keys(), key=len, reverse=True)

print("tag_vocab:", len(tag_vocab))
print("meal_vocab:", len(meal_vocab))
print("feature_vocab:", len(feature_vocab))
print("diet_vocab:", len(diet_vocab))

tag_vocab: 189
meal_vocab: 6
feature_vocab: 39
diet_vocab: 5


In [ ]:
COUNTRY_ALIASES = {
    "uk": "united kingdom",
    "u k": "united kingdom",
    "great britain": "united kingdom",
    "britain": "united kingdom",
    "england": "united kingdom",
    "scotland": "united kingdom",
    "wales": "united kingdom",
    "northern ireland": "united kingdom",
    "deutschland": "germany",
    "espana": "spain",
    "españa": "spain",
    "italia": "italy",
    "grecia": "greece",
}

PRICE_KEYWORDS = {
    "cheap": [
        "cheap",
        "budget",
        "affordable",
        "low cost",
        "low-cost",
        "inexpensive",
        "not expensive",
        "economic",
        "economical",
        "ieftin",
        "ieftina",
        "ieftine",
    ],
    "mid": [
        "mid range",
        "mid-range",
        "moderate",
        "casual",
        "normal price",
        "average price",
        "medium price",
        "mediu",
        "pret mediu",
        "preț mediu",
    ],
    "expensive": [
        "expensive",
        "fine dining",
        "luxury",
        "high end",
        "high-end",
        "premium",
        "fancy",
        "upscale",
        "scump",
        "scumpa",
        "scumpe",
        "lux",
    ],
}

TAG_SYNONYMS = {
    "italian": ["italian", "italian food", "pizza", "pasta", "trattoria", "osteria"],
    "french": ["french", "french food", "bistro", "brasserie"],
    "spanish": ["spanish", "spanish food", "tapas", "paella"],
    "greek": ["greek", "greek food", "taverna", "souvlaki", "gyros"],
    "portuguese": ["portuguese", "portuguese food"],
    "german": ["german", "german food"],
    "seafood": ["seafood", "fish", "oyster", "sushi"],
    "steakhouse": ["steakhouse", "steak", "grill"],
    "asian": ["asian", "chinese", "japanese", "thai", "vietnamese", "korean"],
    "indian": ["indian", "curry"],
    "mexican": ["mexican", "tacos", "burrito"],
    "mediterranean": ["mediterranean"],
    "fast food": ["fast food", "burger", "burgers", "kebab"],
    "coffee": ["coffee", "cafe", "cafes", "café"],
    "bar": ["bar", "pub", "drinks", "cocktails"],
}

DIETARY_SYNONYMS = {
    "vegetarian friendly": ["vegetarian", "veggie", "vegetarian friendly", "fara carne", "fără carne"],
    "vegan options": ["vegan", "vegan options", "plant based", "plant-based"],
    "gluten free options": ["gluten free", "gluten-free", "without gluten", "fara gluten", "fără gluten"],
}

MEAL_SYNONYMS = {
    "breakfast": ["breakfast", "mic dejun"],
    "brunch": ["brunch"],
    "lunch": ["lunch", "pranz", "prânz"],
    "dinner": ["dinner", "cina", "cină"],
    "drinks": ["drinks", "cocktails", "beer", "wine"],
}

FEATURE_SYNONYMS = {
    "reservations": ["reservation", "reservations", "booking", "book a table", "rezervare"],
    "outdoor seating": ["outdoor", "terrace", "terrace seating", "outdoor seating", "terasă", "terasa"],
    "delivery": ["delivery", "takeaway", "take out", "takeout"],
    "wheelchair accessible": ["wheelchair", "accessible", "wheelchair accessible"],
    "family friendly": ["family friendly", "kid friendly", "kids", "children", "familie", "copii"],
    "romantic": ["romantic", "date night", "cozy", "cosy"],
}

In [ ]:
def find_synonym_matches(query_norm, synonym_dict):
    found = []

    for canonical, synonyms in synonym_dict.items():
        for syn in synonyms:
            syn_norm = normalize_text(syn)

            if contains_phrase(query_norm, syn_norm):
                found.append(canonical)
                break

    return sorted(set(found))


def extract_price_bucket(query_norm):
    for bucket, keywords in PRICE_KEYWORDS.items():
        for kw in keywords:
            kw_norm = normalize_text(kw)

            if contains_phrase(query_norm, kw_norm):
                return bucket

    return None


def extract_vocab_matches(query_norm, vocab, max_matches=8, min_len=3):
    found = []

    for term in vocab:
        term_norm = normalize_text(term)

        if len(term_norm) < min_len:
            continue

        if contains_phrase(query_norm, term_norm):
            found.append(term_norm)

        if len(found) >= max_matches:
            break

    return sorted(set(found))


def extract_country(query_norm):
    for alias, canonical in COUNTRY_ALIASES.items():
        alias_norm = normalize_text(alias)

        if contains_phrase(query_norm, alias_norm):
            canonical_norm = normalize_text(canonical)

            if canonical_norm in country_vocab:
                return canonical_norm

    for country in country_vocab:
        if contains_phrase(query_norm, country):
            return country

    return None


def city_has_location_context(query_norm, city_norm):
    if not contains_phrase(query_norm, city_norm):
        return False

    prep_patterns = [
        rf"\bin\s+{re.escape(city_norm)}\b",
        rf"\bnear\s+{re.escape(city_norm)}\b",
        rf"\baround\s+{re.escape(city_norm)}\b",
        rf"\bat\s+{re.escape(city_norm)}\b",
        rf"\bfrom\s+{re.escape(city_norm)}\b",
        rf"\bin apropiere de\s+{re.escape(city_norm)}\b",
        rf"\blanga\s+{re.escape(city_norm)}\b",
        rf"\blângă\s+{re.escape(city_norm)}\b",
        rf"\bin\s+orasul\s+{re.escape(city_norm)}\b",
        rf"\bîn\s+orasul\s+{re.escape(city_norm)}\b",
        rf"\bîn\s+orașul\s+{re.escape(city_norm)}\b",
    ]

    for pattern in prep_patterns:
        if re.search(pattern, query_norm):
            return True

    if len(city_norm) >= 4 and city_norm not in BAD_CITY_TERMS:
        return True

    return False


def extract_city(query_norm):
    for alias, canonical in CITY_ALIASES.items():
        alias_norm = normalize_text(alias)
        canonical_norm = normalize_text(canonical)

        if contains_phrase(query_norm, alias_norm) and canonical_norm in city_vocab:
            return canonical_norm

    candidates = []

    for city in city_vocab:
        if city_has_location_context(query_norm, city):
            candidates.append(city)

    if not candidates:
        return None

    return max(candidates, key=len)


def parse_user_query(query):
    query_norm = normalize_text(query)

    city = extract_city(query_norm)
    country = extract_country(query_norm)
    price_bucket = extract_price_bucket(query_norm)

    tags_from_synonyms = find_synonym_matches(query_norm, TAG_SYNONYMS)
    dietary_from_synonyms = find_synonym_matches(query_norm, DIETARY_SYNONYMS)
    meals_from_synonyms = find_synonym_matches(query_norm, MEAL_SYNONYMS)
    features_from_synonyms = find_synonym_matches(query_norm, FEATURE_SYNONYMS)

    tags_from_vocab = extract_vocab_matches(query_norm, tag_vocab, max_matches=8)
    meals_from_vocab = extract_vocab_matches(query_norm, meal_vocab, max_matches=5)
    features_from_vocab = extract_vocab_matches(query_norm, feature_vocab, max_matches=5)
    diets_from_vocab = extract_vocab_matches(query_norm, diet_vocab, max_matches=5)

    parsed = {
        "original_query": query,
        "normalized_query": query_norm,
        "city": city,
        "country": country,
        "price_bucket": price_bucket,
        "tags": sorted(set(tags_from_synonyms + tags_from_vocab)),
        "dietary": sorted(set(dietary_from_synonyms + diets_from_vocab)),
        "matched_meals": sorted(set(meals_from_synonyms + meals_from_vocab)),
        "matched_features": sorted(set(features_from_synonyms + features_from_vocab)),
    }

    return parsed

In [ ]:
def list_contains_term(row_list, term):
    if not isinstance(row_list, list):
        return False

    term_norm = normalize_text(term)

    if not term_norm:
        return False

    for item in row_list:
        item_norm = normalize_text(item)

        if not item_norm:
            continue

        if term_norm == item_norm:
            return True

        if term_norm in item_norm:
            return True

        if item_norm in term_norm and len(item_norm) >= 4:
            return True

    return False


def list_contains_any(row_list, terms):
    return any(list_contains_term(row_list, term) for term in terms)

In [ ]:
def apply_location_filter(df_in, parsed_query, verbose=True):
    current = df_in.copy()

    city = parsed_query.get("city")
    country = parsed_query.get("country")

    if city:
        before = len(current)
        filtered = current[current["city_filled_norm"] == city].copy()

        if verbose:
            print(f"City filter city={city}: {before} -> {len(filtered)}")

        if len(filtered) == 0:
            if verbose:
                print("Oraș detectat, dar fără rezultate în dataset. Returnăm subset gol.")
            return filtered

        current = filtered

    if country:
        before = len(current)
        filtered = current[current["country_norm"] == country].copy()

        if verbose:
            print(f"Country filter country={country}: {before} -> {len(filtered)}")

        if len(filtered) == 0:
            if verbose:
                print("Țară detectată, dar fără rezultate în dataset. Returnăm subset gol.")
            return filtered

        current = filtered

    return current


def compute_metadata_score_for_row(row, parsed_query):
    score = 0.0
    reasons = []

    price_bucket = parsed_query.get("price_bucket")
    tags = parsed_query.get("tags", [])
    dietary = parsed_query.get("dietary", [])
    meals = parsed_query.get("matched_meals", [])
    features = parsed_query.get("matched_features", [])

    if price_bucket and row.get("price_bucket") == price_bucket:
        score += 2.0
        reasons.append(f"price={price_bucket}")

    row_tags = row.get("top_tags_norm_list", [])
    row_diets = row.get("special_diets_norm_list", [])
    row_meals = row.get("meals_norm_list", [])
    row_features = row.get("features_norm_list", [])

    for tag in tags:
        if list_contains_term(row_tags, tag):
            score += 2.0
            reasons.append(f"tag={tag}")

    for diet in dietary:
        if (
            list_contains_term(row_diets, diet)
            or list_contains_term(row_tags, diet)
            or list_contains_term(row_features, diet)
        ):
            score += 2.0
            reasons.append(f"diet={diet}")

    for meal in meals:
        if list_contains_term(row_meals, meal) or list_contains_term(row_tags, meal):
            score += 1.0
            reasons.append(f"meal={meal}")

    for feature in features:
        if list_contains_term(row_features, feature) or list_contains_term(row_tags, feature):
            score += 1.0
            reasons.append(f"feature={feature}")

    return score, sorted(set(reasons))


def add_metadata_scores(df_in, parsed_query):
    current = df_in.copy()

    if len(current) == 0:
        current["metadata_score"] = pd.Series(dtype=float)
        current["metadata_match_reasons"] = pd.Series(dtype=object)
        return current

    scores_and_reasons = current.apply(
        lambda row: compute_metadata_score_for_row(row, parsed_query),
        axis=1,
    )

    current["metadata_score"] = scores_and_reasons.apply(lambda x: x[0])
    current["metadata_match_reasons"] = scores_and_reasons.apply(lambda x: x[1])

    return current


def apply_metadata_filter(df_in, parsed_query, min_results=50, verbose=True):
    current = add_metadata_scores(df_in, parsed_query)

    if len(current) == 0:
        return current

    has_constraints = bool(
        parsed_query.get("price_bucket")
        or parsed_query.get("tags")
        or parsed_query.get("dietary")
        or parsed_query.get("matched_meals")
        or parsed_query.get("matched_features")
    )

    if not has_constraints:
        if verbose:
            print("No metadata constraints.")
        return current

    before = len(current)
    filtered = current[current["metadata_score"] > 0].copy()

    if verbose:
        print(f"Metadata score > 0 filter: {before} -> {len(filtered)}")

    if len(filtered) >= min_results:
        return filtered.sort_values("metadata_score", ascending=False).reset_index(drop=True)

    if 0 < len(filtered) < min_results and before <= min_results:
        return filtered.sort_values("metadata_score", ascending=False).reset_index(drop=True)

    if verbose:
        print("Metadata filter prea strict. Păstrăm subsetul curent, dar folosim metadata_score la ranking.")

    return current.sort_values("metadata_score", ascending=False).reset_index(drop=True)


def filter_candidates(query, df_in, min_metadata_results=50, verbose=True):
    parsed = parse_user_query(query)

    if verbose:
        print("Parsed query:")
        print(json.dumps(parsed, indent=2, ensure_ascii=False))
        print("\nInitial rows:", len(df_in))

    candidates = apply_location_filter(
        df_in,
        parsed,
        verbose=verbose,
    )

    if verbose:
        print("After location filter:", len(candidates))

    if len(candidates) == 0:
        candidates = add_metadata_scores(candidates, parsed)
        return parsed, candidates.reset_index(drop=True)

    candidates = apply_metadata_filter(
        candidates,
        parsed,
        min_results=min_metadata_results,
        verbose=verbose,
    )

    if verbose:
        print("After metadata filter/scoring:", len(candidates))

    return parsed, candidates.reset_index(drop=True)

In [ ]:
def encode_query(query):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    return query_embedding[0]


def semantic_search_within_candidates(
    query,
    candidates_df,
    embeddings_array,
    top_k=100,
    score_chunk_size=200_000,
    verbose=True,
):
    if len(candidates_df) == 0:
        result = candidates_df.copy()
        result["semantic_score"] = pd.Series(dtype=float)
        return result

    query_embedding = encode_query(query)

    faiss_indices = candidates_df["faiss_idx"].to_numpy(dtype=np.int64)

    assert faiss_indices.min() >= 0
    assert faiss_indices.max() < embeddings_array.shape[0]

    n = len(faiss_indices)
    scores = np.empty(n, dtype=np.float32)

    if verbose:
        print(f"Computing semantic scores for {n:,} candidates...")

    for start in range(0, n, score_chunk_size):
        end = min(start + score_chunk_size, n)

        idx_chunk = faiss_indices[start:end]
        emb_chunk = embeddings_array[idx_chunk]

        scores[start:end] = emb_chunk @ query_embedding

    k = min(top_k, n)

    if k == n:
        top_positions = np.argsort(-scores)
    else:
        top_positions_unsorted = np.argpartition(-scores, k - 1)[:k]
        top_positions = top_positions_unsorted[np.argsort(-scores[top_positions_unsorted])]

    result = candidates_df.iloc[top_positions].copy().reset_index(drop=True)
    result["semantic_score"] = scores[top_positions]

    return result


def retrieve_candidates_for_reranking(
    query,
    df_in,
    embeddings_array,
    candidate_min_metadata_results=50,
    semantic_top_k=100,
    score_chunk_size=200_000,
    verbose=True,
):
    start_time = time.time()

    parsed, candidates = filter_candidates(
        query=query,
        df_in=df_in,
        min_metadata_results=candidate_min_metadata_results,
        verbose=verbose,
    )

    if len(candidates) == 0:
        empty_results = candidates.copy()
        empty_results["semantic_score"] = pd.Series(dtype=float)

        return {
            "query": query,
            "parsed_query": parsed,
            "num_candidates_after_filter": 0,
            "candidates": empty_results,
            "elapsed_seconds": time.time() - start_time,
        }

    semantic_results = semantic_search_within_candidates(
        query=query,
        candidates_df=candidates,
        embeddings_array=embeddings_array,
        top_k=semantic_top_k,
        score_chunk_size=score_chunk_size,
        verbose=verbose,
    )

    elapsed = time.time() - start_time

    return {
        "query": query,
        "parsed_query": parsed,
        "num_candidates_after_filter": len(candidates),
        "candidates": semantic_results,
        "elapsed_seconds": elapsed,
    }

In [ ]:
def normalize_semantic_score(series):

    s = pd.to_numeric(series, errors="coerce").fillna(0.0)
    return ((s + 1.0) / 2.0).clip(0, 1)


def normalize_metadata_score(series):
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)

    max_val = s.max()

    if pd.isna(max_val) or max_val <= 0:
        return pd.Series(np.zeros(len(s)), index=s.index)

    return (s / max_val).clip(0, 1)


def normalize_rating_score(series):

    s = pd.to_numeric(series, errors="coerce")

    normalized = (s / 5.0).clip(0, 1)
    normalized = normalized.fillna(0.5)

    return normalized


def normalize_popularity_score(series):

    s = pd.to_numeric(series, errors="coerce")
    return s.clip(0, 1).fillna(0.5)

In [ ]:
def compute_constraint_scores_for_row(row, parsed_query):

    hard_score = 1.0
    soft_score = 0.0
    max_soft_score = 0.0
    reasons = []

    city = parsed_query.get("city")
    country = parsed_query.get("country")
    price_bucket = parsed_query.get("price_bucket")
    tags = parsed_query.get("tags", [])
    dietary = parsed_query.get("dietary", [])
    meals = parsed_query.get("matched_meals", [])
    features = parsed_query.get("matched_features", [])

    if city:
        if row.get("city_filled_norm") == city:
            reasons.append(f"city={city}")
        else:
            hard_score = 0.0

    if country:
        if row.get("country_norm") == country:
            reasons.append(f"country={country}")
        else:
            hard_score = 0.0

    if price_bucket:
        max_soft_score += 1.0
        if row.get("price_bucket") == price_bucket:
            soft_score += 1.0
            reasons.append(f"price={price_bucket}")

    row_tags = row.get("top_tags_norm_list", [])
    row_diets = row.get("special_diets_norm_list", [])
    row_meals = row.get("meals_norm_list", [])
    row_features = row.get("features_norm_list", [])

    for tag in tags:
        max_soft_score += 1.0
        if list_contains_term(row_tags, tag):
            soft_score += 1.0
            reasons.append(f"tag={tag}")

    for diet in dietary:
        max_soft_score += 1.0
        if (
            list_contains_term(row_diets, diet)
            or list_contains_term(row_tags, diet)
            or list_contains_term(row_features, diet)
        ):
            soft_score += 1.0
            reasons.append(f"diet={diet}")

    for meal in meals:
        max_soft_score += 1.0
        if list_contains_term(row_meals, meal) or list_contains_term(row_tags, meal):
            soft_score += 1.0
            reasons.append(f"meal={meal}")

    for feature in features:
        max_soft_score += 1.0
        if list_contains_term(row_features, feature) or list_contains_term(row_tags, feature):
            soft_score += 1.0
            reasons.append(f"feature={feature}")

    if max_soft_score > 0:
        soft_constraint_score = soft_score / max_soft_score
    else:
        soft_constraint_score = 0.5

    return hard_score, soft_constraint_score, sorted(set(reasons))

In [ ]:
RERANK_WEIGHTS = {
    "semantic": 0.40,
    "metadata": 0.20,
    "rating": 0.15,
    "popularity": 0.10,
    "soft_constraints": 0.15,
}

assert abs(sum(RERANK_WEIGHTS.values()) - 1.0) < 1e-9


def rerank_candidates(candidates_df, parsed_query, top_k=20):
    results = candidates_df.copy()

    if len(results) == 0:
        results["semantic_score_norm"] = pd.Series(dtype=float)
        results["metadata_score_norm"] = pd.Series(dtype=float)
        results["rating_score"] = pd.Series(dtype=float)
        results["popularity_score_norm"] = pd.Series(dtype=float)
        results["hard_constraint_score"] = pd.Series(dtype=float)
        results["soft_constraint_score"] = pd.Series(dtype=float)
        results["constraint_match_reasons"] = pd.Series(dtype=object)
        results["final_rerank_score"] = pd.Series(dtype=float)
        return results

    if "metadata_score" not in results.columns:
        results["metadata_score"] = 0.0

    results["semantic_score_norm"] = normalize_semantic_score(results["semantic_score"])
    results["metadata_score_norm"] = normalize_metadata_score(results["metadata_score"])
    results["rating_score"] = normalize_rating_score(results["rating"])
    results["popularity_score_norm"] = normalize_popularity_score(results["popularity_score"])

    constraint_values = results.apply(
        lambda row: compute_constraint_scores_for_row(row, parsed_query),
        axis=1,
    )

    results["hard_constraint_score"] = constraint_values.apply(lambda x: x[0])
    results["soft_constraint_score"] = constraint_values.apply(lambda x: x[1])
    results["constraint_match_reasons"] = constraint_values.apply(lambda x: x[2])

    results["final_rerank_score"] = (
        RERANK_WEIGHTS["semantic"] * results["semantic_score_norm"]
        + RERANK_WEIGHTS["metadata"] * results["metadata_score_norm"]
        + RERANK_WEIGHTS["rating"] * results["rating_score"]
        + RERANK_WEIGHTS["popularity"] * results["popularity_score_norm"]
        + RERANK_WEIGHTS["soft_constraints"] * results["soft_constraint_score"]
    )

    results["final_rerank_score"] = (
        results["final_rerank_score"] * results["hard_constraint_score"]
    )

    results = results.sort_values(
        [
            "final_rerank_score",
            "soft_constraint_score",
            "metadata_score_norm",
            "semantic_score_norm",
            "rating_score",
            "popularity_score_norm",
        ],
        ascending=False,
    ).reset_index(drop=True)

    if top_k is not None:
        results = results.head(top_k).copy().reset_index(drop=True)

    results.insert(0, "rerank_position", np.arange(1, len(results) + 1))

    return results

In [ ]:
def retrieve_and_rerank(
    query,
    df_in,
    embeddings_array,
    retrieval_top_k=100,
    rerank_top_k=20,
    candidate_min_metadata_results=50,
    verbose=True,
):
    start_time = time.time()

    retrieval_output = retrieve_candidates_for_reranking(
        query=query,
        df_in=df_in,
        embeddings_array=embeddings_array,
        candidate_min_metadata_results=candidate_min_metadata_results,
        semantic_top_k=retrieval_top_k,
        verbose=verbose,
    )

    parsed_query = retrieval_output["parsed_query"]
    candidates = retrieval_output["candidates"]

    reranked = rerank_candidates(
        candidates_df=candidates,
        parsed_query=parsed_query,
        top_k=rerank_top_k,
    )

    elapsed = time.time() - start_time

    return {
        "query": query,
        "parsed_query": parsed_query,
        "num_candidates_after_filter": retrieval_output["num_candidates_after_filter"],
        "num_retrieved_for_reranking": len(candidates),
        "results": reranked,
        "elapsed_seconds": elapsed,
    }

In [ ]:
def preview_reranked_results(results_df, n=10):
    cols = [
        "rerank_position",
        "faiss_idx",
        "restaurant_id",
        "name",
        "city_filled",
        "country",
        "rating",
        "price_bucket",
        "semantic_score",
        "semantic_score_norm",
        "metadata_score",
        "metadata_score_norm",
        "rating_score",
        "popularity_score",
        "popularity_score_norm",
        "soft_constraint_score",
        "hard_constraint_score",
        "final_rerank_score",
        "constraint_match_reasons",
        "metadata_match_reasons",
        "top_tags_text",
        "special_diets_text",
        "meals_text",
        "features_text",
        "short_profile",
    ]

    cols = [c for c in cols if c in results_df.columns]

    return results_df[cols].head(n)


def pretty_print_reranked(query, output, n=5):
    print("=" * 120)
    print("QUERY:", query)
    print("PARSED:")
    print(json.dumps(output["parsed_query"], indent=2, ensure_ascii=False))
    print("Candidates after filter:", output["num_candidates_after_filter"])
    print("Retrieved for reranking:", output["num_retrieved_for_reranking"])
    print("Elapsed seconds:", round(output["elapsed_seconds"], 2))
    print("-" * 120)

    results = output["results"]

    if len(results) == 0:
        print("No results.")
        return

    for i, (_, row) in enumerate(results.head(n).iterrows(), start=1):
        print(f"[{i}] {row.get('name')}")
        print(f"    Location: {row.get('city_filled')}, {row.get('country')}")
        print(f"    Rating: {row.get('rating')}")
        print(f"    Price: {row.get('price_bucket')}")
        print(f"    Final rerank score: {row.get('final_rerank_score'):.4f}")
        print(f"    Semantic norm: {row.get('semantic_score_norm'):.4f}")
        print(f"    Metadata norm: {row.get('metadata_score_norm'):.4f}")
        print(f"    Rating score: {row.get('rating_score'):.4f}")
        print(f"    Popularity score: {row.get('popularity_score_norm'):.4f}")
        print(f"    Soft constraints: {row.get('soft_constraint_score'):.4f}")
        print(f"    Reasons: {row.get('constraint_match_reasons')}")
        print(f"    {row.get('short_profile')}")
        print()

In [ ]:
query = "cheap italian restaurant in rome"

output = retrieve_and_rerank(
    query=query,
    df_in=df,
    embeddings_array=embeddings,
    retrieval_top_k=100,
    rerank_top_k=20,
    verbose=True,
)

pretty_print_reranked(query, output, n=10)
display(preview_reranked_results(output["results"], n=20))

Parsed query:
{
  "original_query": "cheap italian restaurant in rome",
  "normalized_query": "cheap italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=rome: 1033798 -> 12603
After location filter: 12603
Metadata score > 0 filter: 12603 -> 9647
After metadata filter/scoring: 9647
Computing semantic scores for 9,647 candidates...
QUERY: cheap italian restaurant in rome
PARSED:
{
  "original_query": "cheap italian restaurant in rome",
  "normalized_query": "cheap italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
Candidates after filter: 9647
Retrieved for reranking: 100
Elapsed seconds: 13.97
---------------------------------------------------------------------------------

,rerank_position,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,semantic_score_norm,metadata_score,metadata_score_norm,rating_score,popularity_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,constraint_match_reasons,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,1,673752,g187791-d7296290,Pizza & Friends,Rome,Italy,5.0,cheap,0.710352,0.855176,4.0,1.0,1.0,0.202125,0.202125,1.0,1.0,0.862283,"[city=rome, price=cheap, tag=italian]","[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Vegetarian Friendly","Vegetarian Friendly, Vegan Options","Lunch, Dinner",Unknown,"Pizza & Friends | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Vegetarian Friendly"
1,2,666820,g187791-d15835606,Stefano of Rome,Rome,Italy,5.0,cheap,0.721902,0.860951,4.0,1.0,1.0,0.168617,0.168617,1.0,1.0,0.861242,"[city=rome, price=cheap, tag=italian]","[price=cheap, tag=italian]","Cheap Eats, Italian, Romana, Lazio",Unknown,Dinner,"Reservations, Serves Alcohol, Table Service, Seating, Free Wifi","Stefano of Rome | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian, Romana, Lazio"
2,3,672898,g187791-d6364117,Ristorante Pizzeria Andrea,Rome,Italy,4.5,cheap,0.701643,0.850821,4.0,1.0,0.9,0.321450,0.321450,1.0,1.0,0.857474,"[city=rome, price=cheap, tag=italian]","[price=cheap, tag=italian]","Cheap Eats, Italian, Vegetarian Friendly, Vegan Options",Unknown,Unknown,Unknown,"Ristorante Pizzeria Andrea | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Vegetarian Friendly, Vegan Options"
3,4,674464,g187791-d8022283,Sto Bene Roma,Rome,Italy,4.5,cheap,0.719524,0.859762,4.0,1.0,0.9,0.260197,0.260197,1.0,1.0,0.854924,"[city=rome, price=cheap, tag=italian]","[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Fast food","Vegan Options, Vegetarian Friendly",Lunch,Unknown,"Sto Bene Roma | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Pizza, Fast food"
4,5,674088,g187791-d7800254,New Bar,Rome,Italy,5.0,cheap,0.722591,0.861296,4.0,1.0,1.0,0.095702,0.095702,1.0,1.0,0.854088,"[city=rome, price=cheap, tag=italian]","[price=cheap, tag=italian]","Cheap Eats, Italian, Cafe, Deli",Unknown,"Breakfast, Lunch, Dinner, Brunch, Drinks","Free Wifi, Takeout, Reservations, Outdoor Seating, Seating, Street Parking, Television, Wheelchair Accessible, Serves Alcohol, Full Bar, Wine and Beer, Table Service","New Bar | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian, Cafe, Deli"
5,6,668300,g187791-d19944877,Acqua & Farina,Rome,Italy,5.0,cheap,0.704328,0.852164,4.0,1.0,1.0,0.080729,0.080729,1.0,1.0,0.848939,"[city=rome, price=cheap, tag=italian]","[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Romana",Unknown,"Lunch, Breakfast, Dinner, Brunch, Drinks","Reservations, Wine and Beer, Accepts Credit Cards","Acqua & Farina | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Romana"
6,7,665909,g187791-d14095490,Aperiwine,Rome,Italy,5.0,cheap,0.714589,0.857294,4.0,1.0,1.0,0.028243,0.028243,1.0,1.0,0.845742,"[city=rome, price=cheap, tag=italian]","[price=cheap, tag=italian]","Cheap Eats, Italian",Unknown,"Breakfast, Lunch, Dinner, Brunch, Drinks","Reservations, Private Dining, Seating, Street Parking, Wheelchair Accessible, Serves Alcohol, Full Bar, Wine and Beer, Accepts Credit Cards","Aperiwine | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian"
7,8,667114,g187791-d17216285,Food of Roma & India,Rome,Italy,5.0,cheap,0.709223,0.854612,4.0,1.0,1.0,0.022323,0.022323,1.0,1.0,0.844077,"[city=rome, price=cheap, tag=italian]","[price=cheap, tag=italian]","Cheap Eats, Indian, Central-Italian",Unknown,"Breakfast, Lunch, Dinner, Drinks",Unknown,"Food of Roma & India | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Indian, Central-Italian"
8,9,670045,g187791-d2529524,Pizzeria Salernitana,Rome,Italy,4.5,cheap,0.718613,0.859307,

In [ ]:
query = "vegetarian brunch in barcelona"

output = retrieve_and_rerank(
    query=query,
    df_in=df,
    embeddings_array=embeddings,
    retrieval_top_k=100,
    rerank_top_k=20,
    verbose=True,
)

pretty_print_reranked(query, output, n=10)
display(preview_reranked_results(output["results"], n=20))

Parsed query:
{
  "original_query": "vegetarian brunch in barcelona",
  "normalized_query": "vegetarian brunch in barcelona",
  "city": "barcelona",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [
    "brunch"
  ],
  "matched_features": []
}

Initial rows: 1033798
City filter city=barcelona: 1033798 -> 10283
After location filter: 10283
Metadata score > 0 filter: 10283 -> 3713
After metadata filter/scoring: 3713
Computing semantic scores for 3,713 candidates...
QUERY: vegetarian brunch in barcelona
PARSED:
{
  "original_query": "vegetarian brunch in barcelona",
  "normalized_query": "vegetarian brunch in barcelona",
  "city": "barcelona",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [
    "brunch"
  ],
  "matched_features": []
}
Candidates after filter: 3713
Retrieved for reranking: 100
Elapsed seconds: 14.29
--------------------------

,rerank_position,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,semantic_score_norm,metadata_score,metadata_score_norm,rating_score,popularity_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,constraint_match_reasons,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,1,375185,g187497-d12356083,Eixampeling Brunch Café & Bar,Barcelona,Spain,5.0,mid,0.706538,0.853269,3.0,1.0,1.0,0.659077,0.659077,1.0,1.0,0.907215,"[city=barcelona, diet=vegetarian friendly, meal=brunch]","[diet=vegetarian friendly, meal=brunch]","Mid-range, Cafe, Fusion, Healthy","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Brunch, Drinks",Unknown,"Eixampeling Brunch Café & Bar | Barcelona, Spain | rating=5.0 | price=mid | tags=Mid-range, Cafe, Fusion, Healthy"
1,2,382975,g187497-d8531379,Ugot Bruncherie,Barcelona,Spain,4.5,expensive,0.733450,0.866725,3.0,1.0,0.9,0.436266,0.436266,1.0,1.0,0.875317,"[city=barcelona, diet=vegetarian friendly, meal=brunch]","[diet=vegetarian friendly, meal=brunch]","Mid-range, Dessert, Cafe, Mediterranean","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Brunch, Breakfast, Drinks","Highchairs Available, Wheelchair Accessible, Serves Alcohol, Accepts Mastercard, Accepts Visa, Free Wifi, Takeout, Seating, Accepts Credit Cards, Table Service","Ugot Bruncherie | Barcelona, Spain | rating=4.5 | price=expensive | tags=Mid-range, Dessert, Cafe, Mediterranean"
2,3,376240,g187497-d13958571,Bo Kaap,Barcelona,Spain,4.5,expensive,0.624147,0.812073,3.0,1.0,0.9,0.508714,0.508714,1.0,1.0,0.860701,"[city=barcelona, diet=vegetarian friendly, meal=brunch]","[diet=vegetarian friendly, meal=brunch]","Mid-range, Bar, Cafe, Mediterranean","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Dinner, Brunch, Drinks","Takeout, Reservations, Outdoor Seating, Seating, Highchairs Available, Wheelchair Accessible, Serves Alcohol, Full Bar, Free Wifi, Accepts Credit Cards, Table Service, Street Parking, Wine and Beer, Digital Payments, Waterfront","Bo Kaap | Barcelona, Spain | rating=4.5 | price=expensive | tags=Mid-range, Bar, Cafe, Mediterranean"
3,4,380123,g187497-d4133942,Bo de Boqueria,Barcelona,Spain,4.5,expensive,0.626584,0.813292,3.0,1.0,0.9,0.416737,0.416737,1.0,1.0,0.851991,"[city=barcelona, diet=vegetarian friendly, meal=brunch]","[diet=vegetarian friendly, meal=brunch]","Mid-range, Seafood, Mediterranean, Spanish","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner, Brunch, After-hours","Reservations, Outdoor Seating, Seating, Television, Highchairs Available, Serves Alcohol, Full Bar, Wine and Beer, Accepts Mastercard, Accepts Visa, Cash Only, Free Wifi, Accepts Credit Cards, Table Service","Bo de Boqueria | Barcelona, Spain | rating=4.5 | price=expensive | tags=Mid-range, Seafood, Mediterranean, Spanish"
4,5,375507,g187497-d12786067,Carite Bar & Lounge,Barcelona,Spain,5.0,cheap,0.607558,0.803779,3.0,1.0,1.0,0.287341,0.287341,1.0,1.0,0.850246,"[city=barcelona, diet=vegetarian friendly, meal=brunch]","[diet=vegetarian friendly, meal=brunch]","Cheap Eats, Spanish, Gastropub, Vegetarian Friendly","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Dinner, Brunch",Unknown,"Carite Bar & Lounge | Barcelona, Spain | rating=5.0 | price=cheap | tags=Cheap Eats, Spanish, Gastropub, Vegetarian Friendly"
5,6,380405,g187497-d4605459,Bo De Gracia,Barcelona,Spain,4.5,mid,0.656498,0.828249,3.0,1.0,0.9,0.331717,0.331717,1.0,1.0,0.849471,"[city=barcelona, diet=vegetarian friendly, meal=brunch]","[diet=vegetarian friendly, meal=brunch]","Mid-range, Seafood, Mediterranean, European","Vegetarian Friendly, Vegan Options, Gluten Free Options","Drinks, Lunch, Dinner, Brunch, After-hours",Unknown,"Bo De Gracia | Barcelona, Spain | rating=4.5 | price=mid | tags=Mid-range, Seafood, Mediterranean, European"
6,7,376673,g187497-d15007059,Bo

In [ ]:
query = "romantic dinner in paris"

output = retrieve_and_rerank(
    query=query,
    df_in=df,
    embeddings_array=embeddings,
    retrieval_top_k=100,
    rerank_top_k=20,
    verbose=True,
)

pretty_print_reranked(query, output, n=10)
display(preview_reranked_results(output["results"], n=20))

Parsed query:
{
  "original_query": "romantic dinner in paris",
  "normalized_query": "romantic dinner in paris",
  "city": "paris",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [],
  "matched_meals": [
    "dinner"
  ],
  "matched_features": [
    "romantic"
  ]
}

Initial rows: 1033798
City filter city=paris: 1033798 -> 18126
After location filter: 18126
Metadata score > 0 filter: 18126 -> 9307
After metadata filter/scoring: 9307
Computing semantic scores for 9,307 candidates...
QUERY: romantic dinner in paris
PARSED:
{
  "original_query": "romantic dinner in paris",
  "normalized_query": "romantic dinner in paris",
  "city": "paris",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [],
  "matched_meals": [
    "dinner"
  ],
  "matched_features": [
    "romantic"
  ]
}
Candidates after filter: 9307
Retrieved for reranking: 100
Elapsed seconds: 7.07
-------------------------------------------------------------------------------------------

,rerank_position,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,semantic_score_norm,metadata_score,metadata_score_norm,rating_score,popularity_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,constraint_match_reasons,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,1,39866,g187147-d11752078,Jean Yves of Chef Jean-Yves' Table,Paris,France,5.0,expensive,0.510745,0.755373,1.0,1.0,1.0,0.643793,0.643793,0.5,1.0,0.791528,"[city=paris, meal=dinner]",[meal=dinner],"Fine Dining, Dine With a Local Chef, French, Asian",Unknown,Dinner,"Reservations, Digital Payments, Cash Only, Free Wifi","Jean Yves of Chef Jean-Yves' Table | Paris, France | rating=5.0 | price=expensive | tags=Fine Dining, Dine With a Local Chef, French, Asian"
1,2,53581,g187147-d719864,Restaurant de la Tour,Paris,France,4.5,expensive,0.515105,0.757553,1.0,1.0,0.9,0.464038,0.464038,0.5,1.0,0.759425,"[city=paris, meal=dinner]",[meal=dinner],"Mid-range, French, European, Vegetarian Friendly","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner, After-hours","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts American Express, Accepts Mastercard, Accepts Visa, Digital Payments, Accepts Credit Cards, Table Service","Restaurant de la Tour | Paris, France | rating=4.5 | price=expensive | tags=Mid-range, French, European, Vegetarian Friendly"
2,3,45352,g187147-d17609403,"We love italy, Pasta, Pizza & Piadine, Paris",Paris,France,5.0,expensive,0.588022,0.794011,1.0,1.0,1.0,0.145709,0.145709,0.5,1.0,0.757175,"[city=paris, meal=dinner]",[meal=dinner],"Mid-range, Italian, Pizza, Mediterranean",Unknown,"Lunch, Dinner, Drinks",Unknown,"We love italy, Pasta, Pizza & Piadine, Paris | Paris, France | rating=5.0 | price=expensive | tags=Mid-range, Italian, Pizza, Mediterranean"
3,4,47939,g187147-d2221071,Antoine,Paris,France,4.5,expensive,0.525736,0.762868,1.0,1.0,0.9,0.373879,0.373879,0.5,1.0,0.752535,"[city=paris, meal=dinner]",[meal=dinner],"Fine Dining, French",Unknown,"Lunch, Dinner","Reservations, Private Dining, Seating, Valet Parking, Wheelchair Accessible, Serves Alcohol, Wine and Beer, Accepts American Express, Accepts Mastercard, Accepts Visa, Accepts Credit Cards, Table Service","Antoine | Paris, France | rating=4.5 | price=expensive | tags=Fine Dining, French"
4,5,46606,g187147-d19879260,For The Love Of Food,Paris,France,5.0,expensive,0.599103,0.799551,1.0,1.0,1.0,0.062549,0.062549,0.5,1.0,0.751075,"[city=paris, meal=dinner]",[meal=dinner],"Mid-range, French, African, Asian",Unknown,"Lunch, Dinner, Brunch, Drinks",Unknown,"For The Love Of Food | Paris, France | rating=5.0 | price=expensive | tags=Mid-range, French, African, Asian"
5,6,43663,g187147-d15347820,La Part des Ours,Paris,France,5.0,expensive,0.513281,0.756641,1.0,1.0,1.0,0.223721,0.223721,0.5,1.0,0.750028,"[city=paris, meal=dinner]",[meal=dinner],"Mid-range, French",Unknown,"Lunch, Dinner",Unknown,"La Part des Ours | Paris, France | rating=5.0 | price=expensive | tags=Mid-range, French"
6,7,56408,g187147-d9769571,La Table De Philippe,Paris,France,5.0,expensive,0.534560,0.767280,1.0,1.0,1.0,0.158298,0.158298,0.5,1.0,0.747742,"[city=paris, meal=dinner]",[meal=dinner],"Mid-range, French",Unknown,"Lunch, Dinner",Unknown,"La Table De Philippe | Paris, France | rating=5.0 | price=expensive | tags=Mid-range, French"
7,8,44873,g187147-d16853947,Amore da Francesca,Paris,France,5.0,expensive,0.531594,0.765797,1.0,1.0,1.0,0.156312,0.156312,0.5,1.0,0.746950,"[city=paris, meal=dinner]",[meal=dinner],"Mid-range, Italian, Pizza",Unknown,"Dinner, Lunch, Drinks",Unknown,"Amore da Francesca | Paris, France | rating=5.0 | price=expensive | tags=Mid-range, Italian, Pizza"
8,9,47408,g187147-d21221277,To Restaurant,Paris,France,5.0,expensive,0.512104,0.756052,1.0,1.0,1.0,0.148589,0.148589,0.5,1.0,0.742280,"[city=paris, meal=dinner]",[meal=dinner],"Fine Dining, French, Japane

In [ ]:
query = "expensive fine dining in milan"

output = retrieve_and_rerank(
    query=query,
    df_in=df,
    embeddings_array=embeddings,
    retrieval_top_k=100,
    rerank_top_k=20,
    verbose=True,
)

pretty_print_reranked(query, output, n=10)
display(preview_reranked_results(output["results"], n=20))

Parsed query:
{
  "original_query": "expensive fine dining in milan",
  "normalized_query": "expensive fine dining in milan",
  "city": "milan",
  "country": null,
  "price_bucket": "expensive",
  "tags": [
    "fine dining"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=milan: 1033798 -> 8381
After location filter: 8381
Metadata score > 0 filter: 8381 -> 2012
After metadata filter/scoring: 2012
Computing semantic scores for 2,012 candidates...
QUERY: expensive fine dining in milan
PARSED:
{
  "original_query": "expensive fine dining in milan",
  "normalized_query": "expensive fine dining in milan",
  "city": "milan",
  "country": null,
  "price_bucket": "expensive",
  "tags": [
    "fine dining"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
Candidates after filter: 2012
Retrieved for reranking: 100
Elapsed seconds: 2.49
----------------------------------------------------------------------------

,rerank_position,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,semantic_score_norm,metadata_score,metadata_score_norm,rating_score,popularity_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,constraint_match_reasons,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,1,697067,g187849-d8515519,Seta,Milan,Italy,4.5,expensive,0.622156,0.811078,4.0,1.0,0.9,0.654319,0.654319,1.0,1.0,0.874863,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, Mediterranean","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner","Outdoor Seating, Reservations, Private Dining, Seating, Parking Available, Validated Parking, Valet Parking, Wheelchair Accessible, Serves Alcohol, Full Bar, Accepts American Express, Accepts Mastercard, Accepts Visa, Free Wifi, Accepts Credit Cards, Table Service","Seta | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Italian, Seafood, Mediterranean"
1,2,692395,g187849-d1996398,Ba Restaurant,Milan,Italy,4.5,expensive,0.626995,0.813497,4.0,1.0,0.9,0.372630,0.372630,1.0,1.0,0.847662,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Chinese, Seafood, Asian","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner, After-hours",Unknown,"Ba Restaurant | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Chinese, Seafood, Asian"
2,3,689277,g187849-d10464840,LUME,Milan,Italy,4.5,expensive,0.599216,0.799608,4.0,1.0,0.9,0.414231,0.414231,1.0,1.0,0.846266,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, International","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner","Reservations, Private Dining, Seating, Validated Parking, Wheelchair Accessible, Serves Alcohol, Wine and Beer, Accepts American Express, Accepts Mastercard, Accepts Visa, Accepts Credit Cards, Table Service, Parking Available, Valet Parking, Highchairs Available, Full Bar, Free Wifi","LUME | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Italian, Seafood, International"
3,4,690803,g187849-d14110023,L'ALCHIMIA Ristorante & Lounge Bar,Milan,Italy,4.5,expensive,0.609915,0.804957,4.0,1.0,0.9,0.391803,0.391803,1.0,1.0,0.846163,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, Mediterranean","Vegetarian Friendly, Gluten Free Options","Lunch, Dinner, Drinks",Unknown,"L'ALCHIMIA Ristorante & Lounge Bar | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Italian, Seafood, Mediterranean"
4,5,689075,g187849-d10120121,Langosteria Café,Milan,Italy,4.5,expensive,0.607583,0.803792,4.0,1.0,0.9,0.378192,0.378192,1.0,1.0,0.844336,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, Mediterranean",Gluten Free Options,"Lunch, Dinner","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Full Bar, Free Wifi, Accepts Credit Cards, Table Service","Langosteria Café | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Italian, Seafood, Mediterranean"
5,6,692858,g187849-d21068480,Frades Porto Cervo - Milano,Milan,Italy,5.0,expensive,0.600469,0.800235,4.0,1.0,1.0,0.135009,0.135009,1.0,1.0,0.833595,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, Mediterranean",Unknown,"Dinner, Lunch","Reservations, Seating, Table Service","Frades Porto Cervo - Milano | Milan, Italy | rating=5.0 | price=expensive | tags=Fine Dining, Italian, Seafood, Mediterranean"
6,7,691253,g187849-d15617170,Bar All In - Food & Beverage,Milan,Italy,5.0,expensive,0.653148,0.826574,4.0,1.0,1.0,0.005625,0.005625,1.0,1.0,0.831192,"[city=milan, price=expensive, tag=fine dining]","[price=exp

In [ ]:
query = "vegetarian restaurant in berlin"

output = retrieve_and_rerank(
    query=query,
    df_in=df,
    embeddings_array=embeddings,
    retrieval_top_k=100,
    rerank_top_k=20,
    verbose=True,
)

pretty_print_reranked(query, output, n=10)

if len(output["results"]) > 0:
    display(preview_reranked_results(output["results"], n=20))

Parsed query:
{
  "original_query": "vegetarian restaurant in berlin",
  "normalized_query": "vegetarian restaurant in berlin",
  "city": "berlin",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=berlin: 1033798 -> 0
Oraș detectat, dar fără rezultate în dataset. Returnăm subset gol.
After location filter: 0
QUERY: vegetarian restaurant in berlin
PARSED:
{
  "original_query": "vegetarian restaurant in berlin",
  "normalized_query": "vegetarian restaurant in berlin",
  "city": "berlin",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}
Candidates after filter: 0
Retrieved for reranking: 0
Elapsed seconds: 1.96
------------------------------------------------------------------------------------------------------------------------
No results.


In [ ]:
query = "best cheap restaurant in northern ireland"

output = retrieve_and_rerank(
    query=query,
    df_in=df,
    embeddings_array=embeddings,
    retrieval_top_k=100,
    rerank_top_k=20,
    verbose=True,
)

pretty_print_reranked(query, output, n=10)
display(preview_reranked_results(output["results"], n=20))

Parsed query:
{
  "original_query": "best cheap restaurant in northern ireland",
  "normalized_query": "best cheap restaurant in northern ireland",
  "city": null,
  "country": "northern ireland",
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
Country filter country=northern ireland: 1033798 -> 2549
After location filter: 2549
Metadata score > 0 filter: 2549 -> 517
After metadata filter/scoring: 517
Computing semantic scores for 517 candidates...
QUERY: best cheap restaurant in northern ireland
PARSED:
{
  "original_query": "best cheap restaurant in northern ireland",
  "normalized_query": "best cheap restaurant in northern ireland",
  "city": null,
  "country": "northern ireland",
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
Candidates after filter: 517
Retrieved for reranking: 100
Elapsed seconds: 9.15
-------------------------------------

,rerank_position,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,semantic_score_norm,metadata_score,metadata_score_norm,rating_score,popularity_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,constraint_match_reasons,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,1,545138,g190795-d8342616,Michael's Diner,Enniskillen,Northern Ireland,4.5,cheap,0.549067,0.774534,2.0,1.0,0.9,0.750000,0.750000,1.0,1.0,0.869813,"[country=northern ireland, price=cheap]",[price=cheap],"Cheap Eats, Quick Bites, Fast food, British",Vegetarian Friendly,"Breakfast, Lunch, Dinner, Brunch",Unknown,"Michael's Diner | Enniskillen, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Quick Bites, Fast food, British"
1,2,520731,g186482-d12166824,Gwyn's Cafe & Pavilion,Derry,Northern Ireland,4.5,cheap,0.526431,0.763215,2.0,1.0,0.9,0.775756,0.775756,1.0,1.0,0.867862,"[country=northern ireland, price=cheap]",[price=cheap],"Cheap Eats, Quick Bites, Cafe, British","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Brunch, Dinner",Unknown,"Gwyn's Cafe & Pavilion | Derry, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Quick Bites, Cafe, British"
2,3,600026,g667786-d17838328,The Stove,Larne,Northern Ireland,5.0,cheap,0.570664,0.785332,2.0,1.0,1.0,0.526803,0.526803,1.0,1.0,0.866813,"[country=northern ireland, price=cheap]",[price=cheap],"Cheap Eats, Cafe, British",Unknown,"Breakfast, Lunch, Dinner","Reservations, Seating, Table Service, Wheelchair Accessible","The Stove | Larne, Northern Ireland | rating=5.0 | price=cheap | tags=Cheap Eats, Cafe, British"
3,4,520523,g186472-d10643110,The Dessert Bar,Ballycastle,Northern Ireland,5.0,cheap,0.582933,0.791466,2.0,1.0,1.0,0.500000,0.500000,1.0,1.0,0.866587,"[country=northern ireland, price=cheap]",[price=cheap],"Cheap Eats, Dessert, Cafe",Unknown,Unknown,"Takeout, Seating, Outdoor Seating","The Dessert Bar | Ballycastle, Northern Ireland | rating=5.0 | price=cheap | tags=Cheap Eats, Dessert, Cafe"
4,5,596477,g635686-d10108147,O'Briens Junction One,Antrim,Northern Ireland,4.5,cheap,0.529666,0.764833,2.0,1.0,0.9,0.729762,0.729762,1.0,1.0,0.863909,"[country=northern ireland, price=cheap]",[price=cheap],"Cheap Eats, Quick Bites, Cafe, British","Vegetarian Friendly, Vegan Options","Breakfast, Lunch","Takeout, Seating, Parking Available, Free off-street parking, Highchairs Available, Wheelchair Accessible, Free Wifi","O'Briens Junction One | Antrim, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Quick Bites, Cafe, British"
5,6,600036,g667786-d5539648,Silver Lounge,Larne,Northern Ireland,4.5,cheap,0.593454,0.796727,2.0,1.0,0.9,0.592410,0.592410,1.0,1.0,0.862932,"[country=northern ireland, price=cheap]",[price=cheap],"Cheap Eats, Cafe, British, Grill",Vegetarian Friendly,"Breakfast, Lunch, Dinner, Brunch",Unknown,"Silver Lounge | Larne, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Cafe, British, Grill"
6,7,520594,g186474-d4454627,Macari Ice Cream & Coffee,Armagh,Northern Ireland,4.5,cheap,0.535910,0.767955,2.0,1.0,0.9,0.666667,0.666667,1.0,1.0,0.858849,"[country=northern ireland, price=cheap]",[price=cheap],"Cheap Eats, Dessert",Unknown,Unknown,"Takeout, Seating, Wheelchair Accessible","Macari Ice Cream & Coffee | Armagh, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Dessert"
7,8,555228,g2137520-d2483688,The Galley Restaurant,Annalong,Northern Ireland,4.5,cheap,0.562644,0.781322,2.0,1.0,0.9,0.613147,0.613147,1.0,1.0,0.858843,"[country=northern ireland, price=cheap]",[price=cheap],"Cheap Eats, Cafe, Seafood, British","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Dinner, Brunch, After-hours",Unknown,"The Galley Restaurant | Annalong, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Cafe, Seafood, British"
8,9,467037,g1019569-d2200644,Edfield Restaurant,Fivemilet

In [ ]:
test_queries = [
    "cheap italian restaurant in rome",
    "vegetarian brunch in barcelona",
    "romantic dinner in paris",
    "family friendly seafood place in lisbon",
    "coffee and breakfast in athens",
    "expensive fine dining in milan",
    "vegan restaurant near berlin with outdoor seating",
    "tapas place in madrid",
    "family restaurant in london",
    "gluten free restaurant in paris",
]

batch_outputs = {}
batch_rows = []

for q in test_queries:
    print("=" * 120)
    print("Running:", q)

    out = retrieve_and_rerank(
        query=q,
        df_in=df,
        embeddings_array=embeddings,
        retrieval_top_k=100,
        rerank_top_k=20,
        verbose=False,
    )

    batch_outputs[q] = out
    results = out["results"]

    top_row = results.iloc[0] if len(results) > 0 else {}

    batch_rows.append({
        "query": q,
        "parsed_city": out["parsed_query"].get("city"),
        "parsed_country": out["parsed_query"].get("country"),
        "parsed_price_bucket": out["parsed_query"].get("price_bucket"),
        "parsed_tags": out["parsed_query"].get("tags"),
        "parsed_dietary": out["parsed_query"].get("dietary"),
        "parsed_meals": out["parsed_query"].get("matched_meals"),
        "parsed_features": out["parsed_query"].get("matched_features"),
        "num_candidates_after_filter": out["num_candidates_after_filter"],
        "num_retrieved_for_reranking": out["num_retrieved_for_reranking"],
        "num_reranked_results": len(results),
        "top_name": top_row.get("name", None) if len(results) > 0 else None,
        "top_city": top_row.get("city_filled", None) if len(results) > 0 else None,
        "top_country": top_row.get("country", None) if len(results) > 0 else None,
        "top_price": top_row.get("price_bucket", None) if len(results) > 0 else None,
        "top_rating": top_row.get("rating", None) if len(results) > 0 else None,
        "top_final_rerank_score": top_row.get("final_rerank_score", None) if len(results) > 0 else None,
        "top_soft_constraint_score": top_row.get("soft_constraint_score", None) if len(results) > 0 else None,
        "elapsed_seconds": out["elapsed_seconds"],
    })

batch_summary_df = pd.DataFrame(batch_rows)
display(batch_summary_df)

Running: cheap italian restaurant in rome
Running: vegetarian brunch in barcelona
Running: romantic dinner in paris
Running: family friendly seafood place in lisbon
Running: coffee and breakfast in athens
Running: expensive fine dining in milan
Running: vegan restaurant near berlin with outdoor seating
Running: tapas place in madrid
Running: family restaurant in london
Running: gluten free restaurant in paris


,query,parsed_city,parsed_country,parsed_price_bucket,parsed_tags,parsed_dietary,parsed_meals,parsed_features,num_candidates_after_filter,num_retrieved_for_reranking,num_reranked_results,top_name,top_city,top_country,top_price,top_rating,top_final_rerank_score,top_soft_constraint_score,elapsed_seconds
0,cheap italian restaurant in rome,rome,None,cheap,[italian],[],[],[],9647,100,20,Pizza & Friends,Rome,Italy,cheap,5.0,0.862283,1.0,2.361366
1,vegetarian brunch in barcelona,barcelona,None,None,[],[vegetarian friendly],[brunch],[],3713,100,20,Eixampeling Brunch Café & Bar,Barcelona,Spain,mid,5.0,0.907215,1.0,2.619915
2,romantic dinner in paris,paris,None,None,[],[],[dinner],[romantic],9307,100,20,Jean Yves of Chef Jean-Yves' Table,Paris,France,expensive,5.0,0.791528,0.5,2.916507
3,family friendly seafood place in lisbon,lisbon,None,None,[seafood],[],[],[family friendly],286,100,20,Solar 31 - Peixe e Marisco,Lisbon,Portugal,expensive,5.0,0.810324,0.5,4.406113
4,coffee and breakfast in athens,athens,None,None,[coffee],[],[breakfast],[],580,100,20,Coffee Joint,Athens,Greece,cheap,5.0,0.840559,0.5,4.201237
5,expensive fine dining in milan,milan,None,expensive,[fine dining],[],[],[],2012,100,20,Seta,Milan,Italy,expensive,4.5,0.874863,1.0,2.230625
6,vegan restaurant near berlin with outdoor seating,berlin,None,None,[],[vegan options],[],"[outdoor seating, seating]",0,0,0,None,None,None,None,NaN,NaN,NaN,1.967582
7,tapas place in madrid,madrid,None,None,[spanish],[],[],[],5305,100,20,Tapa Tapa Arenal,Madrid,Spain,expensive,4.5,0.888502,1.0,6.326614
8,family restaurant in london,london,None,None,[],[],[],[],0,0,0,None,None,None,None,NaN,NaN,NaN,1.958710
9,gluten free restaurant in paris,paris,None,None,[],[gluten free options],[],[],584,100,20,Restaurant H,Paris,France,expensive,5.0,0.903589,1.0,2.782918


In [ ]:
def compute_location_satisfaction(output, k=10):
    parsed = output["parsed_query"]
    results = output["results"].head(k)

    if len(results) == 0:
        return None

    city = parsed.get("city")
    country = parsed.get("country")

    if city:
        return float((results["city_filled_norm"] == city).mean())

    if country:
        return float((results["country_norm"] == country).mean())

    return None


location_rows = []

for q, out in batch_outputs.items():
    location_rows.append({
        "query": q,
        "parsed_city": out["parsed_query"].get("city"),
        "parsed_country": out["parsed_query"].get("country"),
        "location_satisfaction_at_10": compute_location_satisfaction(out, k=10),
    })

location_metrics_df = pd.DataFrame(location_rows)
display(location_metrics_df)

,query,parsed_city,parsed_country,location_satisfaction_at_10
0,cheap italian restaurant in rome,rome,None,1.0
1,vegetarian brunch in barcelona,barcelona,None,1.0
2,romantic dinner in paris,paris,None,1.0
3,family friendly seafood place in lisbon,lisbon,None,1.0
4,coffee and breakfast in athens,athens,None,1.0
5,expensive fine dining in milan,milan,None,1.0
6,vegan restaurant near berlin with outdoor seating,berlin,None,NaN
7,tapas place in madrid,madrid,None,1.0
8,family restaurant in london,london,None,NaN
9,gluten free restaurant in paris,paris,None,1.0


In [ ]:
def compute_price_satisfaction(output, k=10):
    parsed = output["parsed_query"]
    results = output["results"].head(k)

    if len(results) == 0:
        return None

    price_bucket = parsed.get("price_bucket")

    if not price_bucket:
        return None

    return float((results["price_bucket"] == price_bucket).mean())


price_rows = []

for q, out in batch_outputs.items():
    price_rows.append({
        "query": q,
        "parsed_price_bucket": out["parsed_query"].get("price_bucket"),
        "price_satisfaction_at_10": compute_price_satisfaction(out, k=10),
    })

price_metrics_df = pd.DataFrame(price_rows)
display(price_metrics_df)

,query,parsed_price_bucket,price_satisfaction_at_10
0,cheap italian restaurant in rome,cheap,1.0
1,vegetarian brunch in barcelona,None,NaN
2,romantic dinner in paris,None,NaN
3,family friendly seafood place in lisbon,None,NaN
4,coffee and breakfast in athens,None,NaN
5,expensive fine dining in milan,expensive,1.0
6,vegan restaurant near berlin with outdoor seating,None,NaN
7,tapas place in madrid,None,NaN
8,family restaurant in london,None,NaN
9,gluten free restaurant in paris,None,NaN


In [ ]:
def compute_soft_constraint_satisfaction(output, k=10):
    results = output["results"].head(k)

    if len(results) == 0:
        return None

    if "soft_constraint_score" not in results.columns:
        return None

    return float(results["soft_constraint_score"].mean())


constraint_rows = []

for q, out in batch_outputs.items():
    constraint_rows.append({
        "query": q,
        "mean_soft_constraint_score_at_10": compute_soft_constraint_satisfaction(out, k=10),
    })

constraint_metrics_df = pd.DataFrame(constraint_rows)
display(constraint_metrics_df)

,query,mean_soft_constraint_score_at_10
0,cheap italian restaurant in rome,1.0
1,vegetarian brunch in barcelona,1.0
2,romantic dinner in paris,0.5
3,family friendly seafood place in lisbon,0.5
4,coffee and breakfast in athens,0.5
5,expensive fine dining in milan,1.0
6,vegan restaurant near berlin with outdoor seating,NaN
7,tapas place in madrid,1.0
8,family restaurant in london,NaN
9,gluten free restaurant in paris,1.0


In [ ]:
def add_simple_retrieval_score(results_df):
    baseline = results_df.copy()

    if len(baseline) == 0:
        baseline["simple_retrieval_score"] = pd.Series(dtype=float)
        return baseline

    baseline["semantic_score_norm"] = normalize_semantic_score(baseline["semantic_score"])
    baseline["metadata_score_norm"] = normalize_metadata_score(baseline["metadata_score"])

    baseline["simple_retrieval_score"] = (
        0.75 * baseline["semantic_score_norm"]
        + 0.25 * baseline["metadata_score_norm"]
    )

    baseline = baseline.sort_values("simple_retrieval_score", ascending=False).reset_index(drop=True)

    return baseline


def compare_baseline_vs_rerank(query, k=10):
    retrieval_output = retrieve_candidates_for_reranking(
        query=query,
        df_in=df,
        embeddings_array=embeddings,
        candidate_min_metadata_results=50,
        semantic_top_k=100,
        verbose=False,
    )

    parsed = retrieval_output["parsed_query"]
    candidates = retrieval_output["candidates"]

    baseline = add_simple_retrieval_score(candidates).head(k)
    reranked = rerank_candidates(candidates, parsed, top_k=k)

    return parsed, baseline, reranked


comparison_query = "expensive fine dining in milan"

parsed, baseline_results, reranked_results = compare_baseline_vs_rerank(
    comparison_query,
    k=10,
)

print("Parsed:")
print(json.dumps(parsed, indent=2, ensure_ascii=False))

print("\nBaseline top 10:")
display(preview_reranked_results(baseline_results, n=10))

print("\nReranked top 10:")
display(preview_reranked_results(reranked_results, n=10))

Parsed:
{
  "original_query": "expensive fine dining in milan",
  "normalized_query": "expensive fine dining in milan",
  "city": "milan",
  "country": null,
  "price_bucket": "expensive",
  "tags": [
    "fine dining"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Baseline top 10:


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,semantic_score_norm,metadata_score,metadata_score_norm,popularity_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,690141,g187849-d12631940,My Kitchen Club,Milan,Italy,4.0,expensive,0.673239,0.836620,4.0,1.0,0.018344,"[price=expensive, tag=fine dining]","Fine Dining, Italian, Japanese, Greek",Unknown,Unknown,Unknown,"My Kitchen Club | Milan, Italy | rating=4.0 | price=expensive | tags=Fine Dining, Italian, Japanese, Greek"
1,691115,g187849-d15242530,Into street,Milan,Italy,4.0,expensive,0.659409,0.829705,4.0,1.0,0.058705,"[price=expensive, tag=fine dining]","Fine Dining, Italian, Pizza, Seafood",Unknown,Dinner,Unknown,"Into street | Milan, Italy | rating=4.0 | price=expensive | tags=Fine Dining, Italian, Pizza, Seafood"
2,691253,g187849-d15617170,Bar All In - Food & Beverage,Milan,Italy,5.0,expensive,0.653148,0.826574,4.0,1.0,0.005625,"[price=expensive, tag=fine dining]","Fine Dining, Italian, Pizza",Unknown,"Breakfast, Lunch, Dinner, Drinks","Seating, Full Bar, Wine and Beer","Bar All In - Food & Beverage | Milan, Italy | rating=5.0 | price=expensive | tags=Fine Dining, Italian, Pizza"
3,692412,g187849-d19993762,Dessert Bar Milano,Milan,Italy,4.5,expensive,0.649833,0.824917,4.0,1.0,0.058335,"[price=expensive, tag=fine dining]","Fine Dining, Italian, International",Unknown,Unknown,Unknown,"Dessert Bar Milano | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Italian, International"
4,691248,g187849-d15613458,TABLOID Milano,Milan,Italy,3.5,expensive,0.644643,0.822322,4.0,1.0,0.039029,"[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, International",Unknown,"Dinner, After-hours, Drinks",Unknown,"TABLOID Milano | Milan, Italy | rating=3.5 | price=expensive | tags=Fine Dining, Italian, Seafood, International"
5,689260,g187849-d10433163,Canteen Milano,Milan,Italy,3.5,expensive,0.641496,0.820748,4.0,1.0,0.101900,"[price=expensive, tag=fine dining]","Fine Dining, Mexican, Seafood",Unknown,"Dinner, After-hours, Drinks",Unknown,"Canteen Milano | Milan, Italy | rating=3.5 | price=expensive | tags=Fine Dining, Mexican, Seafood"
6,695966,g187849-d7368809,San Giorgio Bakery & Coffee,Milan,Italy,4.0,expensive,0.638127,0.819064,4.0,1.0,0.031520,"[price=expensive, tag=fine dining]","Fine Dining, Italian, Bar, Pizza",Unknown,"Breakfast, Lunch, Dinner, Drinks","Takeout, Reservations, Outdoor Seating, Seating, Street Parking, Serves Alcohol, Full Bar, Wine and Beer, Digital Payments, Free Wifi, Accepts Credit Cards","San Giorgio Bakery & Coffee | Milan, Italy | rating=4.0 | price=expensive | tags=Fine Dining, Italian, Bar, Pizza"
7,692487,g187849-d2019316,Bon Wei,Milan,Italy,4.0,expensive,0.628973,0.814487,4.0,1.0,0.210357,"[price=expensive, tag=fine dining]","Fine Dining, Chinese, Seafood, Asian","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner",Unknown,"Bon Wei | Milan, Italy | rating=4.0 | price=expensive | tags=Fine Dining, Chinese, Seafood, Asian"
8,692395,g187849-d1996398,Ba Restaurant,Milan,Italy,4.5,expensive,0.626995,0.813497,4.0,1.0,0.372630,"[price=expensive, tag=fine dining]","Fine Dining, Chinese, Seafood, Asian","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner, After-hours",Unknown,"Ba Restaurant | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Chinese, Seafood, Asian"
9,697067,g187849-d8515519,Seta,Milan,Italy,4.5,expensive,0.622156,0.811078,4.0,1.0,0.654319,"[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, Mediterranean","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner","Outdoor Seating, Reservations, Private Dining, Seating, Parking Available, Validated Parking, Valet Parking, Wheelchair Accessible, Serves Alcohol, Full Bar, Accepts American Express, Accepts Mastercard, Accepts Visa, Free Wifi, Accepts Credit Cards, Table Service","Seta | Milan, Italy | 


Reranked top 10:


,rerank_position,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,semantic_score_norm,metadata_score,metadata_score_norm,rating_score,popularity_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,constraint_match_reasons,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,1,697067,g187849-d8515519,Seta,Milan,Italy,4.5,expensive,0.622156,0.811078,4.0,1.0,0.9,0.654319,0.654319,1.0,1.0,0.874863,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, Mediterranean","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner","Outdoor Seating, Reservations, Private Dining, Seating, Parking Available, Validated Parking, Valet Parking, Wheelchair Accessible, Serves Alcohol, Full Bar, Accepts American Express, Accepts Mastercard, Accepts Visa, Free Wifi, Accepts Credit Cards, Table Service","Seta | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Italian, Seafood, Mediterranean"
1,2,692395,g187849-d1996398,Ba Restaurant,Milan,Italy,4.5,expensive,0.626995,0.813497,4.0,1.0,0.9,0.372630,0.372630,1.0,1.0,0.847662,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Chinese, Seafood, Asian","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner, After-hours",Unknown,"Ba Restaurant | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Chinese, Seafood, Asian"
2,3,689277,g187849-d10464840,LUME,Milan,Italy,4.5,expensive,0.599216,0.799608,4.0,1.0,0.9,0.414231,0.414231,1.0,1.0,0.846266,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, International","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner","Reservations, Private Dining, Seating, Validated Parking, Wheelchair Accessible, Serves Alcohol, Wine and Beer, Accepts American Express, Accepts Mastercard, Accepts Visa, Accepts Credit Cards, Table Service, Parking Available, Valet Parking, Highchairs Available, Full Bar, Free Wifi","LUME | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Italian, Seafood, International"
3,4,690803,g187849-d14110023,L'ALCHIMIA Ristorante & Lounge Bar,Milan,Italy,4.5,expensive,0.609915,0.804957,4.0,1.0,0.9,0.391803,0.391803,1.0,1.0,0.846163,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, Mediterranean","Vegetarian Friendly, Gluten Free Options","Lunch, Dinner, Drinks",Unknown,"L'ALCHIMIA Ristorante & Lounge Bar | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Italian, Seafood, Mediterranean"
4,5,689075,g187849-d10120121,Langosteria Café,Milan,Italy,4.5,expensive,0.607583,0.803792,4.0,1.0,0.9,0.378192,0.378192,1.0,1.0,0.844336,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, Mediterranean",Gluten Free Options,"Lunch, Dinner","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Full Bar, Free Wifi, Accepts Credit Cards, Table Service","Langosteria Café | Milan, Italy | rating=4.5 | price=expensive | tags=Fine Dining, Italian, Seafood, Mediterranean"
5,6,692858,g187849-d21068480,Frades Porto Cervo - Milano,Milan,Italy,5.0,expensive,0.600469,0.800235,4.0,1.0,1.0,0.135009,0.135009,1.0,1.0,0.833595,"[city=milan, price=expensive, tag=fine dining]","[price=expensive, tag=fine dining]","Fine Dining, Italian, Seafood, Mediterranean",Unknown,"Dinner, Lunch","Reservations, Seating, Table Service","Frades Porto Cervo - Milano | Milan, Italy | rating=5.0 | price=expensive | tags=Fine Dining, Italian, Seafood, Mediterranean"
6,7,691253,g187849-d15617170,Bar All In - Food & Beverage,Milan,Italy,5.0,expensive,0.653148,0.826574,4.0,1.0,1.0,0.005625,0.005625,1.0,1.0,0.831192,"[city=milan, price=expensive, tag=fine dining]","[price=exp

In [ ]:
aggregate_metrics = {
    "num_test_queries": len(batch_outputs),
    "mean_location_satisfaction_at_10": location_metrics_df["location_satisfaction_at_10"].dropna().mean(),
    "mean_price_satisfaction_at_10": price_metrics_df["price_satisfaction_at_10"].dropna().mean(),
    "mean_soft_constraint_score_at_10": constraint_metrics_df["mean_soft_constraint_score_at_10"].dropna().mean(),
    "mean_elapsed_seconds": batch_summary_df["elapsed_seconds"].mean(),
}

aggregate_metrics_df = pd.DataFrame([aggregate_metrics])
display(aggregate_metrics_df)

,num_test_queries,mean_location_satisfaction_at_10,mean_price_satisfaction_at_10,mean_soft_constraint_score_at_10,mean_elapsed_seconds
0,10,1.0,1.0,0.8125,3.177159


In [ ]:
BATCH_SUMMARY_PATH = OUTPUT_DIR / "reranking_batch_summary.csv"
LOCATION_METRICS_PATH = OUTPUT_DIR / "reranking_location_metrics.csv"
PRICE_METRICS_PATH = OUTPUT_DIR / "reranking_price_metrics.csv"
CONSTRAINT_METRICS_PATH = OUTPUT_DIR / "reranking_constraint_metrics.csv"
AGGREGATE_METRICS_PATH = OUTPUT_DIR / "reranking_aggregate_metrics.csv"

batch_summary_df.to_csv(BATCH_SUMMARY_PATH, index=False)
location_metrics_df.to_csv(LOCATION_METRICS_PATH, index=False)
price_metrics_df.to_csv(PRICE_METRICS_PATH, index=False)
constraint_metrics_df.to_csv(CONSTRAINT_METRICS_PATH, index=False)
aggregate_metrics_df.to_csv(AGGREGATE_METRICS_PATH, index=False)

print("Saved:")
print("-", BATCH_SUMMARY_PATH)
print("-", LOCATION_METRICS_PATH)
print("-", PRICE_METRICS_PATH)
print("-", CONSTRAINT_METRICS_PATH)
print("-", AGGREGATE_METRICS_PATH)

Saved:
- /content/drive/MyDrive/tablewise/artifacts_new/reranking/reranking_batch_summary.csv
- /content/drive/MyDrive/tablewise/artifacts_new/reranking/reranking_location_metrics.csv
- /content/drive/MyDrive/tablewise/artifacts_new/reranking/reranking_price_metrics.csv
- /content/drive/MyDrive/tablewise/artifacts_new/reranking/reranking_constraint_metrics.csv
- /content/drive/MyDrive/tablewise/artifacts_new/reranking/reranking_aggregate_metrics.csv


In [ ]:
example_results_dir = OUTPUT_DIR / "example_reranked_results"
example_results_dir.mkdir(parents=True, exist_ok=True)

for q, out in batch_outputs.items():
    safe_name = normalize_text(q).replace(" ", "_")[:80]
    path = example_results_dir / f"{safe_name}.csv"

    results_to_save = out["results"].copy()
    results_to_save["query"] = q
    results_to_save["parsed_query_json"] = json.dumps(out["parsed_query"], ensure_ascii=False)

    results_to_save.to_csv(path, index=False)

print("Saved reranked example result CSVs in:", example_results_dir)

Saved reranked example result CSVs in: /content/drive/MyDrive/tablewise/artifacts_new/reranking/example_reranked_results


In [ ]:
reranking_config = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "faiss_dir": str(FAISS_DIR),
    "mapping_path": str(MAPPING_PATH),
    "embeddings_path": str(EMBEDDINGS_PATH),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "num_restaurants": int(len(df)),
    "rerank_weights": RERANK_WEIGHTS,
    "score_definitions": {
        "semantic_score_norm": "(cosine_similarity + 1) / 2",
        "metadata_score_norm": "metadata_score / max(metadata_score in candidate set)",
        "rating_score": "rating / 5, missing -> 0.5",
        "popularity_score_norm": "precomputed popularity_score, missing -> 0.5",
        "soft_constraint_score": "fraction of parsed soft constraints matched",
        "hard_constraint_score": "1 if hard location constraints are satisfied, else 0",
        "final_rerank_score": "weighted sum multiplied by hard_constraint_score",
    },
    "location_policy": "city/country are hard filters; no fallback to all Europe if explicit city has zero results",
}

RERANKING_CONFIG_PATH = OUTPUT_DIR / "reranking_config.json"

with open(RERANKING_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(reranking_config, f, indent=2, ensure_ascii=False)

print("Saved reranking config:", RERANKING_CONFIG_PATH)
reranking_config

Saved reranking config: /content/drive/MyDrive/tablewise/artifacts_new/reranking/reranking_config.json


{'created_at': '2026-05-04T17:15:45.139172Z',
 'faiss_dir': '/content/drive/MyDrive/tablewise/artifacts_new/faiss',
 'mapping_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet',
 'embeddings_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'num_restaurants': 1033798,
 'rerank_weights': {'semantic': 0.4,
  'metadata': 0.2,
  'rating': 0.15,
  'popularity': 0.1,
  'soft_constraints': 0.15},
 'score_definitions': {'semantic_score_norm': '(cosine_similarity + 1) / 2',
  'metadata_score_norm': 'metadata_score / max(metadata_score in candidate set)',
  'rating_score': 'rating / 5, missing -> 0.5',
  'popularity_score_norm': 'precomputed popularity_score, missing -> 0.5',
  'soft_constraint_score': 'fraction of parsed soft constraints matched',
  'hard_constraint_score': '1 if hard location constraints are satisfied, else 0',
  'final_rerank_score'

In [ ]:

def tablewise_get_reranked_recommendations(
    query,
    top_k=5,
    verbose=False,
):
    output = retrieve_and_rerank(
        query=query,
        df_in=df,
        embeddings_array=embeddings,
        retrieval_top_k=100,
        rerank_top_k=top_k,
        candidate_min_metadata_results=50,
        verbose=verbose,
    )

    return output


def tablewise_preview_reranked_recommendations(
    query,
    top_k=5,
):
    output = tablewise_get_reranked_recommendations(
        query=query,
        top_k=top_k,
        verbose=True,
    )

    pretty_print_reranked(query, output, n=top_k)
    display(preview_reranked_results(output["results"], n=top_k))

    return output

In [ ]:
final_test_query = "cheap vegetarian italian restaurant in rome"

final_output = tablewise_preview_reranked_recommendations(
    final_test_query,
    top_k=5,
)

print(json.dumps(final_output["parsed_query"], indent=2, ensure_ascii=False))

Parsed query:
{
  "original_query": "cheap vegetarian italian restaurant in rome",
  "normalized_query": "cheap vegetarian italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=rome: 1033798 -> 12603
After location filter: 12603
Metadata score > 0 filter: 12603 -> 10182
After metadata filter/scoring: 10182
Computing semantic scores for 10,182 candidates...
QUERY: cheap vegetarian italian restaurant in rome
PARSED:
{
  "original_query": "cheap vegetarian italian restaurant in rome",
  "normalized_query": "cheap vegetarian italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}
Candidates after filter: 10182
Retrieved for re

,rerank_position,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,semantic_score,semantic_score_norm,metadata_score,metadata_score_norm,rating_score,popularity_score,popularity_score_norm,soft_constraint_score,hard_constraint_score,final_rerank_score,constraint_match_reasons,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,1,663655,g187791-d10546545,Sfiziarte - The Art of Food,Rome,Italy,5.0,cheap,0.721371,0.860686,6.0,1.0,1.0,0.639109,0.639109,1.0,1.0,0.908185,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]","[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Mediterranean, European","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Dinner, Brunch",Unknown,"Sfiziarte - The Art of Food | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian, Mediterranean, European"
1,2,673752,g187791-d7296290,Pizza & Friends,Rome,Italy,5.0,cheap,0.718427,0.859214,6.0,1.0,1.0,0.202125,0.202125,1.0,1.0,0.863898,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]","[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Vegetarian Friendly","Vegetarian Friendly, Vegan Options","Lunch, Dinner",Unknown,"Pizza & Friends | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Vegetarian Friendly"
2,3,673077,g187791-d6578187,Bacio Di Puglia,Rome,Italy,4.5,cheap,0.704145,0.852072,6.0,1.0,0.9,0.359844,0.359844,1.0,1.0,0.861813,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]","[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Fast food, Mediterranean",Vegetarian Friendly,"Lunch, Dinner, Brunch, After-hours",Unknown,"Bacio Di Puglia | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Fast food, Mediterranean"
3,4,674464,g187791-d8022283,Sto Bene Roma,Rome,Italy,4.5,cheap,0.730370,0.865185,6.0,1.0,0.9,0.260197,0.260197,1.0,1.0,0.857094,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]","[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Fast food","Vegan Options, Vegetarian Friendly",Lunch,Unknown,"Sto Bene Roma | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Pizza, Fast food"
4,5,670317,g187791-d3156900,Ristorcaffe Vecchia Roma,Rome,Italy,4.5,cheap,0.737022,0.868511,6.0,1.0,0.9,0.164527,0.164527,1.0,1.0,0.848857,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]","[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Cafe","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Dinner",Unknown,"Ristorcaffe Vecchia Roma | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Pizza, Cafe"


{
  "original_query": "cheap vegetarian italian restaurant in rome",
  "normalized_query": "cheap vegetarian italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}


In [ ]:
FINAL_EXAMPLE_QUERY = "cheap vegetarian italian restaurant in rome"

final_example_output = tablewise_get_reranked_recommendations(
    FINAL_EXAMPLE_QUERY,
    top_k=5,
    verbose=False,
)

FINAL_EXAMPLE_RESULTS_PATH = OUTPUT_DIR / "final_example_top5_for_rag.csv"
FINAL_EXAMPLE_METADATA_PATH = OUTPUT_DIR / "final_example_top5_for_rag_metadata.json"

final_example_output["results"].to_csv(FINAL_EXAMPLE_RESULTS_PATH, index=False)

with open(FINAL_EXAMPLE_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "query": FINAL_EXAMPLE_QUERY,
            "parsed_query": final_example_output["parsed_query"],
            "num_candidates_after_filter": final_example_output["num_candidates_after_filter"],
            "num_retrieved_for_reranking": final_example_output["num_retrieved_for_reranking"],
            "elapsed_seconds": final_example_output["elapsed_seconds"],
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved final RAG example:")
print("-", FINAL_EXAMPLE_RESULTS_PATH)
print("-", FINAL_EXAMPLE_METADATA_PATH)

display(final_example_output["results"])

Saved final RAG example:
- /content/drive/MyDrive/tablewise/artifacts_new/reranking/final_example_top5_for_rag.csv
- /content/drive/MyDrive/tablewise/artifacts_new/reranking/final_example_top5_for_rag_metadata.json


,rerank_position,faiss_idx,restaurant_id,name,country,region,province,city,city_filled,city_filled_norm,city_source,address,latitude,longitude,rating,price_level,price_bucket,top_tags_text,top_tags_norm_list,meals_text,meals_norm_list,features_text,features_norm_list,special_diets_text,special_diets_norm_list,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score,profile_quality_score,short_profile,profile_text,top_tags_list,meals_list,features_list,special_diets_list,country_norm,region_norm,province_norm,city_norm,metadata_score,metadata_match_reasons,semantic_score,semantic_score_norm,metadata_score_norm,rating_score,popularity_score_norm,hard_constraint_score,soft_constraint_score,constraint_match_reasons,final_rerank_score
0,1,663655,g187791-d10546545,Sfiziarte - The Art of Food,Italy,Lazio,None,Rome,Rome,rome,original_city,"Via Andrea Doria 53, 00192 Rome Italy",41.909397,12.451711,5.0,€,cheap,"Cheap Eats, Italian, Mediterranean, European","[cheap eats, italian, mediterranean, european]","Breakfast, Lunch, Dinner, Brunch","[breakfast, lunch, dinner, brunch]",Unknown,[],"Vegetarian Friendly, Vegan Options, Gluten Free Options","[vegetarian friendly, vegan options, gluten free options]",#27 of 10231 Restaurants in Rome,27.0,10231.0,0.639109,11,"Sfiziarte - The Art of Food | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian, Mediterranean, European","Sfiziarte - The Art of Food is a restaurant located in Rome, Lazio, Italy. Address: Via Andrea Doria 53, 00192 Rome Italy. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, Italian, Mediterranean, European. Special diets: Vegetarian Frie...","[Cheap Eats, Italian, Mediterranean, European]","[Breakfast, Lunch, Dinner, Brunch]",[],"[Vegetarian Friendly, Vegan Options, Gluten Free Options]",italy,lazio,,rome,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]",0.721371,0.860686,1.0,1.0,0.639109,1.0,1.0,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]",0.908185
1,2,673752,g187791-d7296290,Pizza & Friends,Italy,Lazio,None,Rome,Rome,rome,original_city,"Piazza Cesare Cantù 5 Metro colli albani, 00179 Rome Italy",41.871307,12.527128,5.0,€,cheap,"Cheap Eats, Italian, Pizza, Vegetarian Friendly","[cheap eats, italian, pizza, vegetarian friendly]","Lunch, Dinner","[lunch, dinner]",Unknown,[],"Vegetarian Friendly, Vegan Options","[vegetarian friendly, vegan options]",#1582 of 10232 Restaurants in Rome,1582.0,10232.0,0.202125,11,"Pizza & Friends | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Vegetarian Friendly","Pizza & Friends is a restaurant located in Rome, Lazio, Italy. Address: Piazza Cesare Cantù 5 Metro colli albani, 00179 Rome Italy. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, Italian, Pizza, Vegetarian Friendly. Special diets: Veg...","[Cheap Eats, Italian, Pizza, Vegetarian Friendly]","[Lunch, Dinner]",[],"[Vegetarian Friendly, Vegan Options]",italy,lazio,,rome,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]",0.718427,0.859214,1.0,1.0,0.202125,1.0,1.0,"[city=rome, diet=vegetarian friendly, price=cheap, tag=italian]",0.863898
2,3,673077,g187791-d6578187,Bacio Di Puglia,Italy,Lazio,None,Rome,Rome,rome,original_city,"Via del Mascherino 59, 00193 Rome Italy",41.905098,12.458700,4.5,€,cheap,"Cheap Eats, Italian, Fast food, Mediterranean","[cheap eats, italian, fast food, mediterranean]","Lunch, Dinner, Brunch, After-hours","[lunch, dinner, brunch, after-hours]",Unknown,[],Vegetarian Friendly,[vegetarian friendly],#368 of 10232 Restaurants in Rome,368.0,10232.0,0.359844,11,"Bacio Di Puglia | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Fast food, Mediterranean","Bacio Di Puglia is a restaurant located in Rome, Lazio, Italy. Address: Via del Mascherino 59, 00193 Rome Italy. Average rating: 4.5 out of 5. Price category: cheap. Original price leve